# 🛠 Setup and Libraries

**1. Install Required Libraries**  
If not already installed, use:
```python
!pip install pandas
!pip install plotly
!pip install seaborn
!pip install matplotlib


In [ ]:

!pip install pandas
!pip install plotly
!pip install seaborn
!pip install matplotlib

# 📥 Load University Data from Google Drive

**1. Import Libraries**  
We import essential Python libraries:  
- `pandas` and `numpy` for data manipulation.  
- `matplotlib.pyplot` and `seaborn` for visualizations.  
- `plotly.express` and `plotly.graph_objects` for interactive plots.  
- `LinearSegmentedColormap` from matplotlib for custom color maps.  

**2. Google Drive CSV Link**  
The `file_id` variable stores the unique ID of the CSV file on Google Drive.  

**3. Construct Download URL**  
We construct a direct download URL using the file ID so pandas can read it directly.  

**4. Read CSV into Pandas**  
`pd.read_csv(csv_url)` loads the data into a DataFrame named `df`.  
`df.head(10)` displays the first 10 rows to inspect the dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import plotly.express as px
import plotly.graph_objects as go

# Google Drive file ID from your link
file_id = "1ii3-HX51DKyhAASESMNqyXCSQLnD_vmz"

# Constructing direct download URL
csv_url = f"https://drive.google.com/uc?id={file_id}&export=download"

# Read the CSV directly into pandas
df = pd.read_csv(csv_url)
df.head(100)


# 🗑 Drop Unnecessary Columns

We remove columns that are not needed for our analysis to clean the dataset:  
- `application_fees` → Not required for current visualizations  
- `exams_accepted` → Already have individual test columns (`GRE`, `IELTS`, etc.)  
- `university_website` → Not needed for analysis  

In [ ]:
df.drop(columns=['application_fees','exams_accepted','university_website'], inplace=True)

# 💰 Parse and Convert Tuition Fees to USD

**1. Import Libraries**  
We use:
- `pandas` and `numpy` for data manipulation.  
- `re` for parsing currency and numeric values from text.  
- `yfinance` to get live currency exchange rates.

**2. Parse Tuition Fees**  
The function `parse_fee` extracts:
- **Currency** (e.g., USD, EUR)  
- **Amount** (numeric value)  
from the `tuition_fees` column.

In [ ]:
import pandas as pd
import numpy as np
import re
import yfinance as yf

# Function to parse tuition fee
def parse_fee(fee):
    match = re.match(r"([A-Za-z$]+)\s*([\d,\.]+)", str(fee))
    if match:
        currency = match.group(1).upper().replace("$", "USD")
        amount = float(match.group(2).replace(",", ""))
        return currency, amount
    return None, np.nan

# Parse tuition fee into currency + amount
df[["currency", "amount"]] = df["tuition_fees"].apply(lambda x: pd.Series(parse_fee(x)))

# Get unique currencies
unique_currencies = df["currency"].dropna().unique()

# Build live exchange rate dictionary
rates = {}
for cur in unique_currencies:
    if cur == "USD":
        rates[cur] = 1.0
    else:
        try:
            ticker = yf.Ticker(f"{cur}USD=X")
            fx = ticker.history(period="1d")["Close"].iloc[-1]
            rates[cur] = fx
        except:
            rates[cur] = np.nan

# Create new column with values in USD
df["tuition_fees_usd"] = df.apply(
    lambda row: row["amount"] * rates.get(row["currency"], np.nan), axis=1
).round(2)  # round to 2 decimals

# Drop helper columns
df = df.drop(columns=["currency", "amount","tuition_fees"])
df


# ⏳ Convert Duration to Months

**1. Define Conversion Function**  
`convert_duration_to_months(duration)` processes the `duration` column:
- Handles missing values (`NaN`)  
- Extracts numeric values using regular expressions  
- Converts years to months if the text contains `"year"`  
- Leaves months as is if the text contains `"month"`


In [ ]:

def convert_duration_to_months(duration):
    if pd.isna(duration):
        return None

    text = duration.lower().strip()

    # Extract numeric value
    num = re.findall(r"[\d\.]+", text)
    if not num:
        return None
    num = float(num[0])

    # If contains "year" → convert to months
    if "year" in text:
        return int(round(num * 12))
    # If contains "month"
    elif "month" in text:
        return int(round(num))
    else:
        return None

df["duration_months"] = df["duration"].apply(convert_duration_to_months)

In [ ]:
df.drop(columns=["duration"], inplace=True)
df.head(10)

# 🧐 Inspect Dataset

**1. Check Data Types and Missing Values**


In [ ]:
print(df.info())          # datatypes + null counts
df.describe(include="all")   # statistics for all columns


# 🔢 Numeric Column Exploration

**1. Identify Numeric Columns**  
We select all columns with numeric data types (`int64` and `float64`) to focus on quantitative analysis.  

**2. Inspect Each Numeric Column**  
For each numeric column, we:  
- Display **summary statistics** (`count`, `mean`, `std`, `min`, `max`, quartiles)  
- Show the **number of unique values** to understand variability and detect potential categorical-like numeric columns  

This step helps in **understanding distributions, spotting outliers, and preparing for further analysis or visualization**.


In [ ]:
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns

for col in numeric_cols:
    print(f"\n📊 Analysis for: {col}")
    print(df[col].describe())
    print("Unique values:", df[col].nunique())


# 🔠 Categorical Column Exploration

**1. Identify Categorical Columns**  
We select all columns with categorical/object data types to focus on qualitative analysis.  

**2. Inspect Each Categorical Column**  
For each categorical column, we:  
- Display the **top 10 most frequent values** to see the dominant categories  
- Show the **number of unique values** to understand diversity and detect high-cardinality columns  

This step helps in **understanding category distributions, spotting anomalies, and preparing for visualizations or encoding for further analysis**.


In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"\n🔠 Analysis for: {col}")
    print(df[col].value_counts().head(10))   # Top 10 frequent values
    print("Unique values:", df[col].nunique())


# 🧹 Handling Missing Values

We start by checking the **number of missing values** in each column using `df.isnull().sum()`, which helps identify incomplete data. Next, we remove all **rows containing missing values** with `df.dropna(inplace=True)` and check the updated dataset shape using `df.shape` to see how many rows remain. After that, we remove any **columns containing missing values** with `df = df.dropna(axis=1)` and again inspect `df.shape` to confirm the new dataset dimensions. This process ensures the dataset is **clean, complete, and ready for analysis or visualization**.


In [ ]:
print(df.isnull().sum())
df.dropna(inplace=True)
df.shape
df = df.dropna(axis=1)
df.shape

# 🎯 Universities Filtered by Test Requirements

- **Normalize Test Columns:** Convert all test columns (`GRE`, `IELTS`, `PTE`, `SAT`, `TOEFL`) to lowercase and strip spaces for consistency.  
- **ONLY GRE:** Filter universities where `GRE = yes` and `IELTS & PTE = no` to get institutions requiring only GRE.  
- **ONLY IELTS:** Filter universities where `IELTS = yes` and `GRE & PTE = no` to get institutions requiring only IELTS.  
- **ONLY SAT:** Filter universities where `SAT = yes` and all other tests (`GRE`, `IELTS`, `PTE`) = no to get institutions requiring only SAT.  
- **ONLY GRE & IELTS:** Filter universities where `GRE = yes` and `IELTS = yes` while `TOEFL & PTE = no` to get institutions requiring both GRE and IELTS.  

This process helps **identify universities by specific test requirements**, analyze patterns, and prepare the dataset for **visualization or reporting**.


In [ ]:
# Normalize Yes/No values
df["IELTS"] = df["IELTS"].str.lower()
df["PTE"] = df["PTE"].str.lower()
df["GRE"] = df["GRE"].str.lower()

# Filter universities where GRE = yes, and IELTS & PTE = no
gre_only = df[
    (df["GRE"] == "yes") &
    (df["IELTS"] == "no") &
    (df["PTE"] == "no")
][["university_name", "subject", "GRE", "IELTS", "PTE"]]

# Show the first 20 results
print("🎯 Universities that require ONLY GRE (no IELTS, no PTE):")
gre_only




In [ ]:
# Filter universities where IELTS = yes, and GRE & PTE = no
ielts_only = df[
    (df["IELTS"] == "yes") &
    (df["GRE"] == "no") &
    (df["PTE"] == "no")
][["university_name", "subject", "GRE", "IELTS", "PTE"]]

print("🎯 Universities that require ONLY IELTS (no GRE, no PTE):")
ielts_only


In [ ]:
# Normalize SAT column
df["SAT"] = df["SAT"].str.lower()  # if SAT has Yes/No values

# Filter universities where SAT = yes and all other tests = no
sat_only = df[
    (df["SAT"] == "yes") &
    (df["GRE"] == "no") &
    (df["IELTS"] == "no") &
    (df["PTE"] == "no")
][["university_name", "subject", "GRE", "IELTS", "PTE", "SAT"]]

print("🎯 Universities that require ONLY SAT (no GRE, no IELTS, no PTE):")
sat_only



In [ ]:
# Normalize Yes/No values
for col in ["GRE", "IELTS", "TOEFL", "PTE"]:
    df[col] = df[col].str.lower().str.strip()

# Filter universities: GRE=Yes, IELTS=Yes, TOEFL=No, PTE=No
only_gre_ielts = df[
    (df["GRE"] == "yes") &
    (df["IELTS"] == "yes") &
    (df["TOEFL"] == "no") &
    (df["PTE"] == "no")
][["university_name", "subject", "GRE", "IELTS", "TOEFL", "PTE"]]

print("🎯 Universities that require ONLY GRE & IELTS (no TOEFL, no PTE):")
only_gre_ielts.head(20)


In [ ]:
df.info()
df.describe(include="all")

# ⏳ Histogram of Program Duration

- **Compute Histogram Data:** Use `np.histogram` on `duration_months` with 10 bins to calculate frequencies.  
- **Define Colors:** Create a list of colors (`green`, `red`, `blue`) to cycle through for the bars.  
- **Plot Bars Manually:** Loop through each bin with `plt.bar`, apply the colors cyclically, and add black edges for clarity.  
- **Customize Plot:** Add a title, axis labels, and grid lines for better readability.  
- **Purpose:** Visualize the distribution of program durations, identify common durations, and enhance readability using color-coding.


In [ ]:

# Compute histogram data
counts, bins = np.histogram(df["duration_months"], bins=10)

# Define colors (cycle green, red, blue)
colors = ["#2ecc71", "#e74c3c", "#3498db"]

# Plot bars manually
for i in range(len(counts)):
    plt.bar(
        bins[i], counts[i],
        width=bins[i+1]-bins[i],
        color=colors[i % len(colors)],  # cycle through colors
        edgecolor="black"
    )
font1 = {'family':'serif','color':'blue','size':20}
plt.title("Distribution of Program Duration (months)", fontsize=14, fontweight="bold",fontdict=font1)
plt.xlabel("Months", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(alpha=0.3)
plt.show()


# 💰 Histogram of Tuition Fees (USD)

- **Define Colors:** Create a list of colors (`pink`, `white`, `red`) to cycle through for the histogram bars.  
- **Plot Histogram:** Use `plt.hist` on `tuition_fees_usd` with 10 bins and add black edges to the bars for clarity.  
- **Color the Bars Manually:** Loop through each bar (`patch`) and assign a color from the list, cycling through if necessary.  
- **Customize Fonts and Labels:** Set the title with a serif font, blue color, and bold weight. Add x-axis label (`Tuition Fees (USD)`), y-axis label (`Frequency`), and grid lines for readability.  
- **Purpose:** Visualize the **distribution of tuition fees** in USD, identify common fee ranges, and enhance visual appeal with **color-coding**.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors to cycle through
colors = ["#ffc0db", "#fff6ff", "#ff0070"]  # pink, white, red

# Plot histogram
n, bins, patches = plt.hist(df["tuition_fees_usd"], bins=10, edgecolor='black')

# Color the bars manually
for i, patch in enumerate(patches):
    patch.set_facecolor(colors[i % len(colors)])  # cycle through colors

# Customize fonts
font1 = {'family':'serif','color':'blue','size':20}
plt.title("Distribution of Tuition Fees (USD)", fontsize=14, fontweight="bold", fontdict=font1)
plt.xlabel("Tuition Fees (USD)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(alpha=0.3)
plt.show()


# 📊 Scatter Plot: Program Duration vs Tuition Fees

- **Create Custom Gradient:** Define a `LinearSegmentedColormap` from red → green → yellow to represent tuition fee intensity.  
- **Initialize Figure:** Set figure size to `(8,6)` for clear visualization.  
- **Plot Scatter Plot:** Plot `duration_months` (x-axis) vs `tuition_fees_usd` (y-axis), with color intensity based on tuition fees, black edges, and slight transparency (`alpha=0.8`).  
- **Customize Fonts and Labels:** Set title with serif font, blue color, bold weight, and increase font size. Add x-axis label (`Duration (Months)`) and y-axis label (`Tuition Fees (USD)`), and include grid lines for better readability.  
- **Add Colorbar Legend:** Include a colorbar showing tuition fee values to indicate color intensity.  
- **Purpose:** Visualize the relationship between **program duration and tuition fees**, detect trends or clusters, and enhance understanding using **color-coded tuition intensity**.


In [ ]:

# Custom gradient: pink → white → red
custom_cmap = LinearSegmentedColormap.from_list("custom", ["red", "green", "yellow"])

plt.figure(figsize=(8,6))

# Scatter plot: duration vs tuition
scatter = plt.scatter(
    df["duration_months"],
    df["tuition_fees_usd"],
    c=df["tuition_fees_usd"],   # color intensity based on tuition
    cmap=custom_cmap,
    edgecolor="black",
    alpha=0.8
)

font1 = {'family':'serif','color':'blue','size':20}

plt.title("Program Duration vs Tuition Fees", fontsize=16, fontweight="bold", fontdict = font1)
plt.xlabel("Duration (Months)", fontsize=16)
plt.ylabel("Tuition Fees (USD)", fontsize=16)
plt.grid(alpha=0.3)

# Add colorbar legend
cbar = plt.colorbar(scatter)
cbar.set_label("Tuition Fees (USD)")

plt.show()


# 📚 Most Frequent Subjects Analysis

- **Top 10 Subjects Overall:** Compute the 10 most frequent subjects across all universities using `value_counts()`.  
- **Plot Overall Subjects:** Create a horizontal bar plot (`sns.barplot`) showing the top 10 subjects with their counts, using the `mako` color palette. Set title, x-axis label (`Count`), and y-axis label (`Subject`).  
- **Most Frequent Subject by Country:** Group the dataset by `country` and find the most common subject in each country. Handle empty counts with `"N/A"`.  
- **Add Counts:** For each country, calculate the number of universities offering its top subject.  
- **Filter Top Countries:** Select the top 10 countries with the largest number of universities for clearer visualization.  
- **Plot Subjects by Country:** Create a bar plot with `country` on the x-axis, `count` on the y-axis, and `subject` as hue. Use `viridis` palette, set title, rotate x-axis labels, add axis labels, and include a legend for the subjects.  
- **Purpose:** Identify **popular subjects globally and by country**, detect trends, and visualize the **distribution of subjects across top countries**.


In [ ]:
# 1️⃣ Most frequent subject overall
most_freq_subject = df["subject"].value_counts().head(10)
print("Top 10 Most Frequent Subjects:")
print(most_freq_subject)

plt.figure(figsize=(10,6))
sns.barplot(
    x=most_freq_subject.values,
    y=most_freq_subject.index,
    palette="mako"
)
plt.title("Top 10 Subjects Across All Universities", fontsize=16, color="blue")
plt.xlabel("Count")
plt.ylabel("Subject")
plt.show()


# Compute the most frequent subject + its count per country
subject_by_country = (
    df.groupby("country")["subject"]
    .agg(lambda x: x.value_counts().index[0] if not x.value_counts().empty else "N/A")
    .reset_index()
)

# Add counts of those top subjects
subject_by_country["count"] = subject_by_country.apply(
    lambda row: (df[(df["country"] == row["country"]) & (df["subject"] == row["subject"])]).shape[0],
    axis=1
)

# Filter for top 10 countries by data size
top_countries = df["country"].value_counts().head(10).index
filtered = subject_by_country[subject_by_country["country"].isin(top_countries)]

print("\n📌 Most Frequent Subject by Country with Counts:")
print(filtered)

# Plot: countries on X-axis, count on Y-axis
plt.figure(figsize=(10,6))
sns.barplot(
    data=filtered,
    x="country",
    y="count",
    hue="subject",   # subject will show in color legend
    dodge=False,
    palette="viridis"
)
plt.title("Most Frequent Subject by Country (Top 10 Countries)", fontsize=16, color="green")
plt.xticks(rotation=45)
plt.ylabel("Count of Top Subject")
plt.xlabel("Country")
plt.legend(title="Top Subject")
plt.show()



# Top 10 Most Expensive university

# 💰 Top 10 Universities by Tuition Fees

- **Sort Universities:** Arrange all universities by `tuition_fees_usd` in **descending order** to identify the highest tuition fees.  
- **Select Top 10:** Extract the top 10 universities with the highest tuition fees.  
- **Display Results:** Show a table with columns: `university_name`, `subject`, `country`, and `tuition_fees_usd` for clarity.  
- **Plot Tuition Fees:** Create a horizontal bar plot (`sns.barplot`) with:  
  - **X-axis:** Tuition Fees (USD).  
  - **Y-axis:** University Name.  
  - **Hue:** Country (to distinguish countries by color).  
- **Customize Style:**  
  - Title: *"Top 10 Universities by Tuition Fees"* in **blue serif font**.  
  - Labels: X → Tuition Fees (USD), Y → University.  
  - Figure size set to `(10,8)` for readability.  
- **Purpose:** Identify which universities charge the **highest tuition fees**, compare across countries, and visualize differences in costs.


In [ ]:
# Sort by tuition fees in descending order
top_10_universities = df.sort_values(by="tuition_fees_usd", ascending=False).head(10)

# Show result
print(top_10_universities[["university_name", "subject", "country", "tuition_fees_usd"]])
font1 = {'family':'serif','color':'blue','size':20}
plt.figure(figsize=(10,8))
sns.barplot(
    data=top_10_universities,
    x="tuition_fees_usd",
    y="university_name",
    hue="country",
    dodge=False
)
plt.title("Top 10 Universities by Tuition Fees", fontdict=font1)
plt.xlabel("Tuition Fees (USD)")
plt.ylabel("University")
plt.show()


# 📉 Top 50 Universities by Lowest Tuition Fees

- **Sort Universities:** Arrange all universities by `tuition_fees_usd` in **ascending order** to identify the lowest tuition fees.  
- **Select Top 50:** Extract the first 50 universities with the **lowest tuition costs**.  
- **Display Results:** Print a table with key columns: `university_name`, `subject`, `country`, and `tuition_fees_usd`.  
- **Plot Tuition Fees:** Use a horizontal bar plot (`sns.barplot`) with:  
  - **X-axis:** Tuition Fees (USD).  
  - **Y-axis:** University Name.  
  - **Hue:** Country (to distinguish universities from different countries).  
- **Customize Style:**  
  - Title: *"Top 50 Universities by Low Tuition Fees"* in **blue serif font**.  
  - Labels: X → Tuition Fees (USD), Y → University.  
  - Figure size set to `(10,8)` for better readability.  
- **Purpose:** Highlight universities offering **affordable tuition**, making it easier to identify cost-effective study options across countries.


In [ ]:
# Sort by tuition fees in Ascending order
top_10_universities = df.sort_values(by="tuition_fees_usd", ascending=True).head(50)

# Show result
print(top_10_universities[["university_name", "subject", "country", "tuition_fees_usd"]])
font1 = {'family':'serif','color':'blue','size':20}
plt.figure(figsize=(10,8))
sns.barplot(
    data=top_10_universities,
    x="tuition_fees_usd",
    y="university_name",
    hue="country",
    dodge=False
)
plt.title("Top 50 Universities by low Tuition Fees", fontdict=font1)
plt.xlabel("Tuition Fees (USD)")
plt.ylabel("University")
plt.show()




# 🏛️ Distribution of University Establishment Years

- **Convert Column:** Transform the `year_of_establish` column into numeric values using `pd.to_numeric` with `errors="coerce"` to handle invalid entries.  
- **Prepare Visualization:** Create a histogram plot with figure size `(10,5)` for better clarity.  
- **Plot Distribution:** Use `sns.histplot` with:  
  - Data: `year_of_establish`.  
  - Bins: 30 (to group years into ranges).  
  - KDE: Disabled (`kde=False`) to only show frequency.  
  - Color: Green bars for visualization.  
- **Customize Style:**  
  - Title: *"Distribution of University Establishment Years"* in **blue serif font**.  
  - X-axis: Year of Establishment.  
  - Y-axis: Number of Universities.  
- **Purpose:** Understand how universities are distributed across different historical periods, identifying trends in establishment over time.


In [ ]:
df["year_of_establish"] = pd.to_numeric(df["year_of_establish"], errors="coerce")
plt.figure(figsize=(10,5))
sns.histplot(df["year_of_establish"], bins=30, kde=False,color='green')
font1 = {'family':'serif','color':'blue','size':20}
plt.title("Distribution of University Establishment Years", fontdict=font1)
plt.xlabel("Year of Establishment")
plt.ylabel("Number of Universities")
plt.show()



# 🔥 Correlation Heatmap of Numerical Columns

- **Identify Numerical Columns:** Select all columns with data types `int64` or `float64` and store them in `numerical_cols`.  
- **Identify Categorical Columns:** Select all columns with data type `object` and store them in `categorical_cols`.  
- **Check Condition:** If there are more than one numerical column, proceed with correlation analysis.  
- **Prepare Visualization:** Create a heatmap with figure size `(10,8)` for clear readability.  
- **Plot Heatmap:** Use `sns.heatmap` with:  
  - Data: Correlation matrix of `numerical_cols`.  
  - `annot=True`: Display correlation values on each cell.  
  - `cmap="coolwarm"`: Color scheme to highlight positive (red) and negative (blue) correlations.  
  - `fmt=".2f"`: Format correlation values to two decimal places.  
- **Customize Style:** Add title *"Correlation Heatmap of Numerical Columns"* with font size `16`.  
- **Purpose:** Identify linear relationships between numerical features, detect redundancy, and find strongly correlated variables.


In [ ]:
# Numerical columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

# Categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

if len(numerical_cols) > 1:
    plt.figure(figsize=(10,8))
    sns.heatmap(df[numerical_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap of Numerical Columns", fontsize=16)
    plt.show()



# 📊 Tuition Fees Distribution Analysis (Boxplots)

### Tuition Fees: Public vs Private Universities
- **Check Column Availability:** Ensure `tuition_fees_usd` exists in the dataset.  
- **Create Boxplot:**  
  - X-axis → `university_type` (Public / Private).  
  - Y-axis → `tuition_fees_usd`.  
  - Palette → `"coolwarm"` for visual contrast.  
- **Customize Style:**  
  - Title → *"Tuition Fees Distribution (Public vs Private)"* with font size `14`.  
  - Labels → X-axis: *University Type*, Y-axis: *Tuition Fees (USD)*.  
- **Purpose:** Compare tuition fee distribution between Public and Private universities, spot outliers, and observe variability.  

---



In [ ]:
if "tuition_fees_usd" in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x="university_type", y="tuition_fees_usd", data=df, palette="coolwarm")
    plt.title("Tuition Fees Distribution (Public vs Private)", fontsize=14)
    plt.xlabel("University Type")
    plt.ylabel("Tuition Fees (USD)")
    plt.show()




### Tuition Fees Distribution by Country
- **Check Column Availability:** Ensure both `tuition_fees_usd` and `country` exist in the dataset.  
- **Create Boxplot:**  
  - X-axis → `country`.  
  - Y-axis → `tuition_fees_usd`.  
  - Rotate X-ticks by `45°` for readability.  
- **Customize Style:**  
  - Title → *"Tuition Fees Distribution by Country"* with font size `16`.  
  - Labels → X-axis: *Country*, Y-axis: *Tuition Fees (USD)*.  
- **Purpose:** Visualize tuition fee variation across different countries, detect outliers, and compare ranges.  

In [ ]:
if "tuition_fees_usd" in df.columns and "country" in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x="country", y="tuition_fees_usd", data=df)
    plt.xticks(rotation=45)

    plt.title("Tuition Fees Distribution by Country", fontsize=16)
    plt.ylabel("Tuition Fees (USD)")
    plt.xlabel("Country")
    plt.show()


# 📝 Test Requirements by University Type

### 1. Data Preparation
- Grouped dataset by **`university_type`** (e.g., Public vs Private).  
- For each test in `["GRE", "IELTS", "TOEFL", "PTE", "SAT", "GMAT"]`, counted how many universities require it (`yes`).  
- Generated a pivot-like table summarizing standardized test requirements for each university type.  

### 2. Visualization
- **Heatmap** (Seaborn):  
  - **X-axis** → Standardized tests.  
  - **Y-axis** → University type.  
  - **Cell values** → Number of universities requiring the test.  
  - **Color scheme** → `"Blues"` to highlight intensity of requirements.  

### 3. Style & Customization
- Annotated cells with actual counts (`fmt="d"`).  
- Added chart title → *"Test Requirements by University Type"* (`fontsize=14`).  
- Axis labels:  
  - X → *Test*  
  - Y → *University Type*  

### 4. Purpose
This visualization helps compare **which tests are more commonly required by public vs private universities**, making it easier for applicants to plan their test strategy.  


In [ ]:
# Define test columns
test_cols = ["GRE", "IELTS", "TOEFL", "PTE", "SAT", "GMAT"]

# --- 1️⃣ Prepare heatmap data
# Count how many universities of each type require each test
heatmap_data = df.groupby("university_type")[test_cols].apply(lambda x: (x=="yes").sum())

# --- 2️⃣ Plot heatmap
plt.figure(figsize=(8,6))
sns.heatmap(
    heatmap_data,        # data
    annot=True,          # show numbers
    cmap="Blues",        # color palette
    fmt="d"              # integer formatting
)

# --- 3️⃣ Customize chart
plt.title("Test Requirements by University Type", fontsize=16, fontweight="bold", color="blue")
plt.ylabel("University Type", fontsize=12)
plt.xlabel("Test", fontsize=12)

# --- 4️⃣ Display
plt.show()

# 📝 Distribution of Test Requirement Combinations

### 1. Data Preparation
- Created a new feature by **combining all test requirement values** (GRE, IELTS, TOEFL, PTE, SAT, GMAT) into a single string for each university.  
- Example → `"yes-no-yes-no-no-no"`.  
- Counted unique combinations across all universities using `.value_counts()`.  

### 2. Visualization
- **Bar Chart** (Seaborn):  
  - **X-axis** → Unique test requirement combinations.  
  - **Y-axis** → Number of universities following that combination.  
  - Used `"mako"` color palette for better contrast.  

### 3. Style & Customization
- Rotated X-axis labels (`45°`) with right alignment for readability.  
- Added chart title → *"Distribution of Test Requirement Combinations"* (`fontsize=14`).  
- Axis labels:  
  - X → *Test Requirement Combination*  
  - Y → *Number of Universities*  

### 4. Purpose
This visualization highlights **how universities group standardized test requirements together**, showing which **combinations of tests** are most common across institutions.  


In [ ]:
# Count unique combinations of tests
combo_counts = df[test_cols].apply(lambda row: "-".join(row.values), axis=1).value_counts()

plt.figure(figsize=(10,5))
sns.barplot(x=combo_counts.index, y=combo_counts.values, palette="mako")
plt.xticks(rotation=45, ha="right")
plt.title("Distribution of Test Requirement Combinations", fontsize=14)
plt.ylabel("Number of Universities")
plt.xlabel("Test Requirement Combination")
plt.show()


# 📊 University Test Requirements Analysis (Plotly)

**1. Data Normalization**  
The test columns (`GRE`, `IELTS`, `TOEFL`, `PTE`, `SAT`, `GMAT`) are converted from *Yes/No* into *1/0* values for easy counting and analysis.  

**2. Test Requirement Counts**  
A **bar chart** shows how many universities require each test.  

**3. Heatmap by University Type**  
A **heatmap** compares test requirements between **public** and **private** universities, highlighting differences.  

**4. Top Test Requirement Combinations**  
The **Top 10 combinations** of test requirements are visualized in another bar chart to see common patterns.  

**5. Interactive Visualizations**  
All charts are **interactive**, allowing zooming, hovering, and panning for quick insights into university admission test trends.


In [ ]:
import plotly.express as px

test_cols = ["GRE", "IELTS", "TOEFL", "PTE", "SAT", "GMAT"]

# --- 1. Normalize values (yes/no -> 1/0)
for col in test_cols:
    df[col] = df[col].map(lambda x: 1 if str(x).lower() == "yes" else 0)

# --- 2. Count of universities requiring each test
test_counts = df[test_cols].sum().reset_index()
test_counts.columns = ["Test", "Count"]

fig1 = px.bar(
    test_counts,
    x="Test", y="Count",
    color="Test",
    title="Universities Requiring Each Test"
)

# --- 3. Heatmap: Test requirements by University Type
heatmap_data = df.groupby("university_type")[test_cols].sum().reset_index()
heatmap_data = heatmap_data.melt(id_vars="university_type", var_name="Test", value_name="Count")

fig2 = px.imshow(
    heatmap_data.pivot(index="university_type", columns="Test", values="Count"),
    color_continuous_scale="Blues",
    title="Test Requirements by University Type"
)

# --- 4. Test requirement combinations (Top 10)
combo_counts = df[test_cols].astype(str).agg("-".join, axis=1).value_counts().head(10).reset_index()
combo_counts.columns = ["Combination", "Count"]

fig3 = px.bar(
    combo_counts,
    x="Combination", y="Count",
    color="Combination",
    title="Top 10 Test Requirement Combinations"
)
fig3.update_xaxes(tickangle=45)

# Show interactive plots
fig1.show()
fig2.show()
fig3.show()


In [ ]:
df.to_csv("golab university dataset.csv", index=False)